In [7]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append(os.path.abspath(".."))  

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import numpy as np
from models.transformer.model import FeatureTokenizer

np.random.seed(0)

# Simulate a small batch: 2 samples, each with 3 features (like 3 columns of tabular data)
batch, num_features, d_model = 2, 3, 4
x = np.random.randn(batch, num_features)

tok = FeatureTokenizer(num_features=num_features, d_model=d_model)
tokens = tok(x)

print("x shape:", x.shape)            # (2, 3)
print("tokens shape:", tokens.shape)  # (2, 3, 4) -- each feature becomes 1 token of dimension d_model
print(tokens[0])                      # the 3 tokens of the first sample


x shape: (2, 3)
tokens shape: (2, 3, 4)
[[ 2.37022999 -0.3775979  -0.2575049   1.02433928]
 [ 0.08151537  0.82298465  0.43067715  0.06885683]
 [ 0.61437087  0.4618535   2.06802138 -0.28396869]]


In [12]:
def loss_fn(out):
    return 0.5 * np.sum(out ** 2), out  # dL/dout = out

def numerical_grad(f, arr, eps=1e-4):
    grad = np.zeros_like(arr)
    it = np.nditer(arr, flags=["multi_index"])
    for _ in it:
        idx = it.multi_index
        orig = arr[idx]
        arr[idx] = orig + eps; plus = f()
        arr[idx] = orig - eps; minus = f()
        arr[idx] = orig
        grad[idx] = (plus - minus) / (2 * eps)
    return grad

def forward_loss():
    out = tok(x)
    loss, _ = loss_fn(out)
    return loss

out = tok(x)
_, grad_out = loss_fn(out)
grad_x_analytic = tok.backward(grad_out)
grad_x_numeric = numerical_grad(forward_loss, x)

err = np.max(np.abs(grad_x_analytic - grad_x_numeric) / (np.abs(grad_x_analytic) + np.abs(grad_x_numeric) + 1e-8))
print(f"max relative error: {err:.2e}  ({'OK' if err < 1e-3 else 'FAIL'})")


max relative error: 2.28e-12  (OK)


In [4]:
# ## 2. Full Transformer demo on TEXT data (word-level) - toy sentiment classification
#
# In part 1, `FeatureTokenizer` only works for NUMERIC tabular data (each column is 1 continuous value, column order carries no meaning). This part uses **`TextClassifierTransformer`** (in `models/transformer/model.py`) - the same full encoder-decoder architecture (self-attention + cross-attention + feed-forward, each branch with a residual + LayerNorm) as `TabularTransformer`, but the input tokenizing layer changes to:
#
# - **`TokenEmbedding`** (`utils/layers/embedding.py`): looks up an embedding table by WORD INDEX in the vocabulary, instead of an affine projection of a numeric value.
# - **`PositionalEncoding`**: adds POSITION information (fixed sin/cos) to the embedding - mandatory for text since word order carries meaning (unlike column order in tabular data, which has no meaning, so `TabularTransformer` doesn't need it).
#
# Demo task: classify a short sentence as "positive" or "negative" (toy sentiment classification) - the encoder learns context between words via self-attention, the decoder uses 1 query token (like `[CLS]`) that cross-attends to pool the whole sentence into 1 vector, then `Linear + Sigmoid` produces a probability, trained with `BCELoss`.

PAD, UNK = 0, 1

# Toy dataset: short sentences labeled positive (1) / negative (0)
train_sentences = [
    "i love this movie", "this is great", "what a wonderful day",
    "i am very happy", "amazing job team", "this food is delicious",
    "i hate this movie", "this is bad", "what a terrible day",
    "i am very sad", "awful job team", "this food is disgusting",
]
train_labels = np.array([[1], [1], [1], [1], [1], [1], [0], [0], [0], [0], [0], [0]], dtype=float)

# Build a vocabulary from the whole training set: each unique word -> 1 integer index
# <pad>=0 pads sentences shorter than max_len; <unk>=1 is for unknown words not in the vocab.
all_words = sorted(set(w for s in train_sentences for w in s.split()))
vocab = {"<pad>": PAD, "<unk>": UNK}
for w in all_words:
    vocab[w] = len(vocab)
vocab_size = len(vocab)
max_len = max(len(s.split()) for s in train_sentences)


def encode(sentence, vocab, max_len):
    """Sentence (string) -> list of token ids, truncated/padded to max_len."""
    ids = [vocab.get(w, UNK) for w in sentence.split()]
    ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
    return ids


X_train = np.array([encode(s, vocab, max_len) for s in train_sentences])

print("vocab_size:", vocab_size, "(unique words + <pad> + <unk>)")
print("max_len:", max_len, "(number of words in the longest training sentence)")
print("X_train shape:", X_train.shape)
print(f"First sentence: \"{train_sentences[0]}\" -> token ids: {X_train[0]}")

# Reading the result: `X_train shape (12, 4)` - 12 sentences, each padded to exactly 4 tokens (sentences shorter than 4 words get `0` (`<pad>`) at the end). Each number in `token ids` is that word's index in `vocab`; e.g. the first sentence "i love this movie" (4 words, no padding needed) has 4 distinct indices, none equal to `0`.

vocab_size: 26 (unique words + <pad> + <unk>)
max_len: 4 (number of words in the longest training sentence)
X_train shape: (12, 4)
First sentence: "i love this movie" -> token ids: [14 17 22 18]


In [5]:
from models.transformer.model import TextClassifierTransformer
from utils.loss.BCELoss import BCELoss
from utils.optimizers.Adam import Adam


def make_pad_mask(token_ids):
    # (batch, seq_len) -> (batch, 1, 1, seq_len): 1 at real word positions, 0 at
    # <pad> positions -- broadcasts against (batch, heads, seq_q, seq_k) inside
    # MultiHeadAttention to stop attention from "looking at" the padding.
    return (token_ids != PAD)[:, np.newaxis, np.newaxis, :].astype(float)


np.random.seed(0)
model = TextClassifierTransformer(
    vocab_size=vocab_size, d_model=16, num_heads=2, d_ff=32, num_layers=2, max_len=max_len
)
loss_fn = BCELoss()
optimizer = Adam(model.parameters(), lr=0.01)

train_mask = make_pad_mask(X_train)

n_epochs = 200
loss_history = []
for epoch in range(n_epochs):
    optimizer.zero_grad()

    pred = model(X_train, src_mask=train_mask)
    loss = loss_fn(pred, train_labels)
    loss_history.append(loss)

    grad_loss = loss_fn.backward()
    model.backward(grad_loss)
    optimizer.step()

    if epoch % 40 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:4d} — loss: {loss:.4f}")

# Reading the result: with a toy dataset of only 12 sentences and 2 clearly separated classes (the "positive" and "negative" vocabularies barely overlap), the loss usually drops very fast toward 0 (the model easily memorizes this training set) - the goal here is to verify the mechanism (embedding + positional encoding + encoder + decoder + backward) works correctly end-to-end, not to evaluate a real NLP benchmark.

Epoch    0 — loss: 2.0587
Epoch   40 — loss: 0.0065
Epoch   80 — loss: 0.0011
Epoch  120 — loss: 0.0009
Epoch  160 — loss: 0.0007
Epoch  199 — loss: 0.0006


In [6]:
# NEW sentences, never seen verbatim in the training set, but composed of
# words already in the vocab -- check whether the model generalizes.
test_sentences = [
    "i love this food",        # expected: positive
    "this movie is terrible",  # expected: negative
    "amazing wonderful day",   # expected: positive
    "this team is bad",        # expected: negative
]
X_test = np.array([encode(s, vocab, max_len) for s in test_sentences])
test_mask = make_pad_mask(X_test)
probs = model(X_test, src_mask=test_mask)

for s, p in zip(test_sentences, probs):
    label = "positive" if p[0] >= 0.5 else "negative"
    print(f"\"{s}\" -> prob={p[0]:.4f} -> {label}")

# Reading the result: all 4 test sentences are NEW combinations of known words (e.g. "food" only appeared in positive sentences during training, but "movie"/"team" appear in both classes) - if the model predicts the right direction (sentences 1 & 3 with high probability near 1, sentences 2 & 4 with low probability near 0), that's evidence `TextClassifierTransformer` isn't just "memorizing" specific sentences, but has learned **each word's embedding** (e.g. clustering "love/amazing/wonderful/great/delicious" close together in embedding space, separate from "hate/terrible/bad/awful/disgusting") - exactly the spirit of self-attention: understanding a SENTENCE's meaning based on the CONTEXT of its words, not just pattern-matching the whole sentence.

"i love this food" -> prob=0.9995 -> positive
"this movie is terrible" -> prob=0.0006 -> negative
"amazing wonderful day" -> prob=0.9995 -> positive
"this team is bad" -> prob=0.0006 -> negative
